THIS IS RAG APPLICATION WITH LOCAL FILES AND GROQ LLM


In [ ]:
from nt import environ
from dotenv import load_dotenv
import os


load_dotenv()
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ['HF_TOKEN'] = os.getenv('HF_TOKEN')
environ

In [ ]:
from langchain_community.document_loaders import Docx2txtLoader

loader = Docx2txtLoader("PutYourDOCUMENT.docx")
docs = loader.load()

content = docs[0].page_content

content




In [ ]:
#Sentence Transformer
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

reuslt = embeddings.embed_query("What is the name of the person in the resume?")

len(reuslt)







In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


split=RecursiveCharacterTextSplitter(chunk_size=400,chunk_overlap=200)
data=split.split_documents(docs)

data







In [ ]:
from langchain_community.vectorstores import Chroma
db=Chroma.from_documents(
    documents=data, 
    embedding=embeddings,
    persist_directory="./chroma_db"
)

#db.query_embeddings("What is the name of the person in the resume?")

In [ ]:
db.similarity_search("What is the name of the person skills?")

In [ ]:
retriever=db.as_retriever()




In [ ]:
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage

llm=ChatGroq(model_name="openai/gpt-oss-120b", api_key= os.getenv("GROQ_API_KEY"))



In [ ]:
result=llm.invoke("Tell me a joke")

In [ ]:
result.content

In [ ]:
#For Retriveral chain, llm chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

prompt=ChatPromptTemplate.from_template(
    """
Answer the following question based only on the provided context:
<context>
{context}
</context>


Question:
{input}


"""
)

document_chain=create_stuff_documents_chain(llm,prompt)
document_chain

HERE DOCUMENT CHAIN AND RETRIEVEL CHAIN NEED TO COMBINE

In [ ]:
from langchain_classic.chains import create_retrieval_chain
retrieval_chain=create_retrieval_chain(retriever,document_chain)

In [ ]:
result=retrieval_chain.invoke({"input": "does phython mentioned?"})
result
#result['answer']

In [ ]:
result=retrieval_chain.invoke({"input": "WHose resume it is? and what are the skills"})
result

In [ ]:
result=retrieval_chain.invoke({"input": "Does this resume fit for AI Lead?"})
result

In [ ]:
from langchain_core.documents import Document
document_chain.invoke({
    "input":"LangSmith has two usage limits: total traces and extended",
    "context":[Document(page_content="LangSmith has two usage limits: total traces and extended traces. These correspond to the two metrics we've been tracking on our usage graph. ")]
})

User question
     ↓
Retriever (search)
     ↓
Documents
     ↓
Document Chain (LLM + prompt)
     ↓
Final answer

In [ ]:
from streamlit_jupyter import StreamlitPatcher
import streamlit as st

# This must be run BEFORE any other st. commands
StreamlitPatcher().jupyter()
query = st.text_input("What you want to know about resume")
if query:
    response = retrieval_chain.invoke({"input" : query})
    st.write(response)